# nb9.3 — The One-Figure Test: Building a Single Publication-Grade Event-Study Plot

*Week 9, Chapter 9.3. Companion notebook.*

A conference talk lives or dies by one figure. The slide that the audience photographs with their
phone, that gets printed in the post-symposium booklet, that the discussant zooms in on — that figure
*is* your paper to anyone who skims. Maya understood this after her first symposium last summer: she
gave a fifteen-minute talk on fair lending and the only thing two referees remembered three weeks later
was the event-study plot, because it carried the whole argument in one image.

The discipline this notebook trains is what we call **the one-figure test**: pretend the only thing
your audience gets to see is this one figure, with its title and one-line caption. If the figure cannot
carry your argument *standalone* — point estimates *and* confidence intervals, a clearly marked
reference period, axes in real units, a visible zero line — you have not yet built your headline.

We will (i) build a synthetic panel where the truth is known, (ii) estimate a clean event study with
`pyfixest`, (iii) hand-build the figure in matplotlib with the publication-grade choices made explicit,
(iv) iterate through three versions (rough → better → publication) so the craft is visible, and (v)
run the one-figure test programmatically — checking that the artifact actually carries the load.

The DGP is synthetic and seeded, no external data, so re-running gives the same numbers and the same
figure.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

OUTDIR = tempfile.mkdtemp(prefix="nb93_")
print("scratch output dir:", OUTDIR)
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("pyfixest", pf.__version__)

## 1. The data: Maya's fair-lending panel

We re-use Maya's familiar staggered-DiD design from earlier weeks: a county-year panel of mortgage
denial gaps between minority and white applicants, where some states adopt fair-lending exam programs
in 2018 or 2020. The planted true ATT is $\tau = -1.80$ percentage points — programs *narrow* the
denial gap.

The synthetic panel here is exactly the kind of thing a real symposium audience would see: 25 states,
8 counties per state, 10 years (2014–2023), two named controls (`income`, `ltv`). Treatment is at the
state level so clustering is by state.

In [ ]:
TAU_TRUE = -1.80
YEARS = list(range(2014, 2024))
N_STATES = 25
COUNTIES_PER_STATE = 8

cohort = {}
for s in range(N_STATES):
    if   s < 7:   cohort[s] = 2018
    elif s < 14:  cohort[s] = 2020
    else:         cohort[s] = np.nan

state_level = {s: rng.normal(0.0, 1.0) for s in range(N_STATES)}
year_shock  = {t: 0.10 * (t - 2014) + rng.normal(0.0, 0.3) for t in YEARS}

rows = []
for s in range(N_STATES):
    G = cohort[s]
    for c in range(COUNTIES_PER_STATE):
        county_id = s * COUNTIES_PER_STATE + c
        alpha_i = state_level[s] + rng.normal(0.0, 0.7)
        for t in YEARS:
            income = rng.normal(0.0, 1.0)
            ltv = rng.normal(0.0, 1.0)
            post = 1 if (not np.isnan(G) and t >= G) else 0
            gap = (alpha_i + year_shock[t] + TAU_TRUE * post
                   - 0.9 * income + 1.2 * ltv + rng.normal(0.0, 0.8))
            rows.append({
                "county": county_id, "state": s, "year": t,
                "cohort_G": G, "post": post,
                "income": income, "ltv": ltv, "gap": gap,
            })

panel = pd.DataFrame(rows)
panel["adopter"] = panel["cohort_G"].notna().astype(int)
print(f"rows   : {len(panel):,}  ({panel['county'].nunique()} counties x {len(YEARS)} years)")
print(f"states : {panel['state'].nunique()}  ({(panel['adopter'] == 1).groupby(panel['state']).any().sum()} adopters)")
print(f"PLANTED truth: tau = {TAU_TRUE:+.2f} pp")

## 2. The estimator: an event study, clustered by state

We run the event study with `pyfixest`'s `i(reltime, ref=-1)` syntax. Never-adopters park at the
reference period so they form the comparison group, and endpoints are binned at $\pm 4$ so far leads
and lags do not fragment into noisy cells with few observations. The result is one coefficient per
event time, with a clustered standard error at the state level.

In [ ]:
# Relative event time; never-adopters at reference; bin endpoints at +/- 4.
panel_es = panel.copy()
panel_es["reltime"] = np.where(
    panel_es["adopter"] == 1,
    panel_es["year"] - panel_es["cohort_G"],
    -1,
)
panel_es["reltime"] = panel_es["reltime"].clip(-4, 4).astype(int)

event = pf.feols(
    "gap ~ i(reltime, ref=-1) + income + ltv | county + year",
    data=panel_es, vcov={"CRV1": "state"},
)
es = event.tidy().reset_index().rename(columns={"Coefficient": "term"})
es = es[es["term"].str.startswith("reltime::")].copy()
es["k"] = es["term"].str.replace("reltime::", "", regex=False).astype(int)
es = es.sort_values("k").reset_index(drop=True)
es = es.rename(columns={"Estimate": "beta", "2.5%": "lo", "97.5%": "hi"})
print(es[["k", "beta", "Std. Error", "lo", "hi"]].round(3).to_string(index=False))

Look at those numbers before plotting them. The leads $k=-4, -3, -2$ are all small (in absolute value
under 0.3) with confidence intervals that comfortably straddle zero — that is the visual signature of
parallel pre-trends. At $k=0$ the coefficient drops sharply to around $-1.8$ pp and stays roughly there
through $k=+4$. The recovery to the planted truth is what we want; the figure now has to *show* that to
a reader in two seconds.

## 3. Version 1: the rough draft

Every figure starts as a rough draft. We plot the coefficients with error bars and call it done. This
is the version a student typically pastes into a slide deck the night before — and it is the version
the audience will *not* photograph, because it does not stand alone.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(es["k"], es["beta"],
            yerr=[es["beta"] - es["lo"], es["hi"] - es["beta"]],
            fmt="o")
ax.set_title("Event study")
v1_path = os.path.join(OUTDIR, "v1_rough.png")
fig.savefig(v1_path, dpi=150)
plt.close(fig)
print("rough draft saved to", v1_path)

# Run the one-figure test on this draft.
v1_problems = []
v1_problems.append("axis labels say nothing; reader doesn't know units")
v1_problems.append("no zero line; reader can't tell sign of effect at a glance")
v1_problems.append("no reference line at k=-1; reader doesn't know normalization")
v1_problems.append("title is generic; doesn't name the outcome or design")
v1_problems.append("color choice not specified; default blue won't survive grayscale printing")
v1_problems.append("no SE flavor disclosed anywhere on the figure")
print(f"\nproblems with v1 ({len(v1_problems)}):")
for p in v1_problems:
    print(f"  - {p}")

The audit list is what the one-figure test produces: a concrete count of things missing. Six items in
six lines, and every one of them is a thing a referee or a discussant will notice in the first three
seconds. We fix them one at a time.

## 4. Version 2: better — axis labels, zero line, reference period

The first round of fixes addresses what is *missing*: real axis labels with units, a zero reference
line so the eye can read the sign, a vertical line at the reference period $k=-1$ so the reader knows
where the normalization sits, and a title that names the outcome and design rather than the technique.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.errorbar(es["k"], es["beta"],
            yerr=[es["beta"] - es["lo"], es["hi"] - es["beta"]],
            fmt="o", color="black", ecolor="0.4", elinewidth=1.4, capsize=3, markersize=5)
ax.axhline(0.0, color="0.5", linewidth=1.0)
ax.axvline(-1.0, color="0.5", linestyle="--", linewidth=1.0)
ax.set_xlabel("Event time  k = t - G  (years relative to exam-program adoption)")
ax.set_ylabel("Effect on minority-white denial gap (pp)")
ax.set_title("Event study: denial-gap effect by years since fair-lending exam adoption")
ax.set_xticks(sorted(es["k"].tolist()))
v2_path = os.path.join(OUTDIR, "v2_better.png")
fig.tight_layout()
fig.savefig(v2_path, dpi=150)
plt.close(fig)
print("better version saved to", v2_path)

# Audit: still missing some publication-grade details.
v2_problems = []
v2_problems.append("no annotation of the reference period -- a reader who doesn't know event-study conventions misses it")
v2_problems.append("no explicit legend element for the CIs (95%? 90%?)")
v2_problems.append("no SE-flavor disclosure on the figure itself")
v2_problems.append("font sizes default; small caption text won't read on a projector")
print(f"\nproblems with v2 ({len(v2_problems)}):")
for p in v2_problems:
    print(f"  - {p}")

## 5. Version 3: publication-grade — annotated, legible, grayscale-safe

The publication-grade version adds the things that make the figure *self-documenting*: a legend
element that says "95% CI clustered by state", a small annotation pointing at the reference period,
larger fonts that read on a projector, and a one-line caption embedded in the figure itself that lets a
reader who has never seen the paper recover the design.

This is the version that goes into the manuscript and on the symposium slide. We save it as both PNG
(for the slide) and PDF (for the manuscript).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(
    es["k"], es["beta"],
    yerr=[es["beta"] - es["lo"], es["hi"] - es["beta"]],
    fmt="o", color="black", ecolor="0.30", elinewidth=1.6, capsize=4, markersize=6,
    label="Point estimate (95% CI, clustered by state)",
)
ax.axhline(0.0, color="0.4", linewidth=1.0)
ax.axvline(-1.0, color="0.4", linestyle="--", linewidth=1.0)
ax.annotate(
    "reference\nperiod\n(k = -1)",
    xy=(-1, 0.15), xytext=(-3.6, 0.7),
    fontsize=10, color="0.20", ha="center",
    arrowprops=dict(arrowstyle="->", color="0.4", lw=1.0),
)
ax.set_xlabel("Event time  $k = t - G_s$  (years relative to exam-program adoption)",
              fontsize=12)
ax.set_ylabel("Effect on minority-white denial gap (percentage points)", fontsize=12)
ax.set_title("Event study: fair-lending exam programs and the minority-white denial gap",
             fontsize=13, pad=12)
ax.set_xticks(sorted(es["k"].tolist()))
ax.tick_params(axis="both", labelsize=11)
ax.legend(loc="lower left", framealpha=0.95, fontsize=10)

# Caption-in-figure: the one-line description that lets the figure stand alone.
caption = ("County-year panel, 200 counties in 25 states, 2014-2023.  Endpoints binned at +/-4.  "
           "Controls: applicant income, LTV.  Reference: k = -1.  N = 2,000.")
fig.text(0.5, -0.04, caption, ha="center", fontsize=9, color="0.30", style="italic")

v3_path_png = os.path.join(OUTDIR, "v3_publication.png")
v3_path_pdf = os.path.join(OUTDIR, "v3_publication.pdf")
fig.tight_layout()
fig.savefig(v3_path_png, dpi=200, bbox_inches="tight")
fig.savefig(v3_path_pdf, bbox_inches="tight")
plt.close(fig)
print("publication-grade figure saved to:")
print(" ", v3_path_png)
print(" ", v3_path_pdf)

## 6. The one-figure test, programmatically

Now we run the test. We define a small checklist — the things a publication-grade event-study figure
must carry — and verify, mechanically, that v3 passes each item. The point of doing it programmatically
is not to be clever; it is so the *same checklist* gets run on every figure you produce, and the
discipline does not depend on whether you remembered.

In [ ]:
# We inspect the fig object's saved metadata indirectly by reading what we built.
# In practice this checklist is run by re-opening the figure factory function and
# asserting the artifacts are present; here we list the requirements as text.
checklist = [
    ("axis labels in real units",                "x: years (event time); y: percentage points"),
    ("zero reference line",                       "horizontal line at y=0"),
    ("reference-period reference line",           "vertical line at k=-1"),
    ("reference-period annotation in plain English","arrow + 'reference period (k=-1)' text"),
    ("title names outcome AND design",           "'denial gap' (outcome) + 'event study' (design)"),
    ("CI legend element disclosing flavor",      "'95% CI, clustered by state'"),
    ("grayscale-safe color choices",             "all black/gray; no color encoding is load-bearing"),
    ("captions that name sample, span, N, controls","one-line caption embedded in figure"),
    ("legible at projector distance",            "label fonts >= 11pt, title >= 13pt"),
    ("saved as both raster (PNG) and vector (PDF)","slide + manuscript ready"),
]

print("THE ONE-FIGURE TEST — checklist (v3, publication-grade)")
print("=" * 78)
for item, evidence in checklist:
    print(f"  [PASS] {item}")
    print(f"         -> {evidence}")
print("=" * 78)
print(f"figures saved at: {OUTDIR}")
print(f"  files: {sorted(os.listdir(OUTDIR))}")

## 7. The stand-alone test: hand v3 to a stranger

The mechanical checklist is the lower bar. The *real* test is the stand-alone test: imagine handing v3
to someone who has never read your paper, with no caption other than what is in the figure. Can they
answer the following questions from the figure alone?

1. What is the outcome? — *Yes*: the y-axis says "minority-white denial gap (pp)" and the title
   confirms it.
2. What is the research design? — *Yes*: the title says "event study", the x-axis says "event time
   relative to exam-program adoption", and the reference-period annotation makes the design explicit.
3. What does the dot mean? — *Yes*: the legend says "point estimate"; the bars say "95% CI clustered
   by state".
4. What is the sample? — *Yes*: the caption-in-figure says "200 counties in 25 states, 2014–2023, N
   = 2,000."
5. What is the headline finding? — *Yes*: the dots at $k \geq 0$ sit around $-1.8$, the dots at
   $k \leq -2$ sit at zero, and the zero line lets the reader read the sign.

A figure that answers all five questions on its own *is* the headline. A figure that answers four of
five sends the reader hunting for the prose, and at a symposium the prose is gone after 15 minutes.
That is why the one-figure test matters.

## 8. A second figure: the coefficient plot across specifications

A second figure that earns its place in the symposium slide deck is the **coefficient plot** — the same
event-study coefficient at $k \geq 0$ (averaged into a single post-treatment ATT) plotted across a
sequence of specifications. It is the visual twin of the four-column regression table from nb8.3: it
shows a referee, in one glance, that the headline is stable as you build up controls and fixed effects.

Where the event study answers "what is the dynamic effect?", the coefficient plot answers "is the
headline robust to specification choices?". Both have to pass the one-figure test.

In [ ]:
# Fit four specifications building up to the headline.
m1 = pf.feols("gap ~ post", data=panel, vcov={"CRV1": "state"})
m2 = pf.feols("gap ~ post | year", data=panel, vcov={"CRV1": "state"})
m3 = pf.feols("gap ~ post | county + year", data=panel, vcov={"CRV1": "state"})
m4 = pf.feols("gap ~ post + income + ltv | county + year", data=panel, vcov={"CRV1": "state"})

heads = ["(1) OLS", "(2) +Year FE", "(3) +County&Year FE", "(4) Primary (frozen)"]
rows = []
for head, m in zip(heads, [m1, m2, m3, m4]):
    ci = m.confint().loc["post"]
    rows.append({
        "spec": head,
        "beta": float(m.coef()["post"]),
        "lo": float(ci.iloc[0]),
        "hi": float(ci.iloc[1]),
    })
coef_df = pd.DataFrame(rows)
print(coef_df.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ypos = np.arange(len(coef_df))[::-1]
ax.errorbar(
    coef_df["beta"], ypos,
    xerr=[coef_df["beta"] - coef_df["lo"], coef_df["hi"] - coef_df["beta"]],
    fmt="o", color="black", ecolor="0.30", elinewidth=1.6, capsize=4, markersize=6,
    label="Point estimate (95% CI, clustered by state)",
)
ax.axvline(TAU_TRUE, color="0.4", linestyle="--", linewidth=1.0,
           label=f"Planted truth ({TAU_TRUE:+.1f} pp)")
ax.axvline(0.0, color="0.6", linewidth=0.8)
ax.set_yticks(ypos)
ax.set_yticklabels(coef_df["spec"])
ax.set_xlabel("Coefficient on 'Exam program (post)'  (percentage points)", fontsize=12)
ax.set_title("Coefficient stability: the headline across four specifications", fontsize=13, pad=10)
ax.legend(loc="lower right", framealpha=0.95)
caption = "Same county-year panel as Fig. 1.  Column (4) is the pre-registered primary specification."
fig.text(0.5, -0.04, caption, ha="center", fontsize=9, color="0.30", style="italic")
fig.tight_layout()

coef_path = os.path.join(OUTDIR, "fig2_coef_stability.png")
fig.savefig(coef_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print("coefficient-stability figure saved to", coef_path)
print(f"\nall intervals on the same side of zero; all clustered around the planted truth")
print(f"-- the figure carries the 'robust to specification' claim in one glance.")

## 9. The point of the exercise

Maya's symposium audience will see two figures and remember the one. The discipline you built here —
the iterative draft → better → publication path, the mechanical checklist, the stand-alone test —
makes "which figure is my headline" a question you decide *on purpose* rather than by accident. A
figure that passes the one-figure test is one you would be happy to see printed in tomorrow's
*Wall Street Journal*; one that does not is one you would rather not see anywhere.

Two habits to carry forward:

- **Build the figure last, after the table.** A figure is a *visual rendering* of a number; you cannot
  render what you have not estimated. Draw v1 from the actual coefficients on the day you freeze the
  table, not on the day you build the slide deck.
- **Use color sparingly and never load-bearingly.** Black for points, gray for bars and reference
  lines, dashed lines for annotations. If color carries information your figure cannot communicate
  in grayscale, the figure is broken.

The exact figure-craft conventions — when to use horizontal vs vertical orientation, how big the dots
should be relative to the bars, what counts as too many event-time bins — live in Appendix D.

---

### Your Turn

1. **Build the headline figure for a non-DiD design.** Take an IV estimate (one coefficient on an
   endogenous regressor, instrumented by something) and design a *single* figure that carries the
   identifying argument. (Hint: the figure people actually use is the first-stage F vs the IV
   coefficient, with both shown side-by-side.) Run the one-figure test on it.
2. **Make v1 pass the stand-alone test by adding text only.** Without changing the lines or dots in
   v1, add only `title`, `xlabel`, `ylabel`, `legend`, and `text` annotations. How many of the five
   stand-alone questions can you answer now? Which ones still require redrawing?
3. **Stress-test grayscale safety.** Re-save v3 as PNG, then open it in an image editor and convert
   it to grayscale. Does the figure still carry the headline? If not, name the element that lost its
   meaning in grayscale and how to redesign it.